Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Retrieval augmented generation

- A model only knows what is in its prompt and what it saw in training
- RAG puts the right documents into the prompt before the question is asked
- Split, embed, retrieve, then answer from what came back

Builds a small store and answers from the retrieved text rather than from
training data.

### Exercise
1. Analyze the code below that uses a vector store and RAG.
2. Replace the source documents and the query text in the last line.
What embeddings actually do
An embedding turns a piece of text into a list of numbers (a vector) that captures its meaning. Texts about similar things end up with similar vectors, even when they share no words. "What language do the Dutch speak?" and "The official language of the Netherlands is Dutch" have almost no words in common, but their embeddings sit close together because they mean related things. That "closeness in meaning" is the whole trick.

In [2]:
%pip install -q langchain-ollama faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\languages\Python311\python.exe -m pip install --upgrade pip


In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Model : chat on the OpenAI-compatible Ollama bridge
llm = make_llm()

# Source documents
docs = [  # Replace the source documents with your own
    "LangChain is a framework for working with LLMs.",
    "RAG combines context matching with answer generation.",
    "FAISS is a library for storing and searching embeddings."
]

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
splits = splitter.create_documents(docs)

# Embeddings + vector store : embeddings ALSO run on Ollama
embeddings = make_embeddings()
vectorstore = FAISS.from_documents(splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Prompt RAG
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only based on the context:\n{context}"),
    ("user", "{question}")
])

# Pipeline
rag_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke({"question": "What is FAISS?"}))

C:\Users\wille\AppData\Local\Temp\ipykernel_71564\3674303122.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS is a library for storing and searching.


### Solution

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Model : chat on the OpenAI-compatible Ollama bridge
llm = make_llm()
# Source documents : replaced with your own
docs = [
    "Expenses over 50 euro need a receipt attached.",
    "The Netherlands has about 17.5 million inhabitants.",
    "The official language of the Netherlands is Dutch.",
    "The Netherlands is known for its tulips, windmills, and cycling culture.",
    "The currency used in the Netherlands is the euro."
]

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
splits = splitter.create_documents(docs)

# Embeddings + vector store : embeddings ALSO run on Ollama
embeddings = make_embeddings()
vectorstore = FAISS.from_documents(splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Prompt RAG
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only based on the context:\n{context}"),
    ("user", "{question}")
])

# Pipeline
rag_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke({"question": "What language is spoken in the Netherlands?"}))  # <- changed question

### Other Solution
Why not feed every document to the model? For our tiny example, five short sentences, we could. Skip FAISS entirely and paste everything into the prompt: This works and gives the right answer. For five sentences, embeddings are overkill.

context = "\n".join(docs)
prompt = f"Answer based on this context:\n{context}\n\nQuestion: What language is spoken in the Netherlands?"
print(llm.invoke(prompt).content)

The problem is scale. Real RAG is a 300-page manual, a company wiki, thousands of documents. Two walls hit us:
The context window. A model can only read so much text at once (qwen3.5:4b maybe a few thousand tokens by default). A 300-page document doesn't fit. We physically cannot paste it all in.
Cost and noise. Even if it fit, sending 300 pages for every question is slow, and burying the one relevant paragraph in 300 pages of irrelevant text makes the model's answer worse.
What embeddings buy us
They let us find the few relevant pieces without reading everything. The process:

Split the big document into chunks, embed each one once, store the vectors (that's FAISS).
When a question comes in, embed the question.
Compare the question's vector to all the chunk vectors and grab the 2-3 closest ones.
Send only those to the model.
So instead of "read all 300 pages," it's "read the 3 paragraphs that match the question." The embedding is what makes step 3, finding the matches by meaning, possible.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Model : chat on the OpenAI-compatible Ollama bridge
llm = make_llm()

# Source documents
docs = [
    "Expenses over 50 euro need a receipt attached.",
    "The Netherlands has about 17.5 million inhabitants.",
    "The official language of the Netherlands is Dutch.",
    "The Netherlands is known for its tulips, windmills, and cycling culture.",
    "The currency used in the Netherlands is the euro."
]

# Prompt : the entire document list goes straight into the context
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only based on the context:\n{context}"),
    ("user", "{question}")
])

# Pipeline : no retriever; just join all docs into the context
chain = (
    {
        "context": lambda x: "\n".join(docs),      # all documents, read directly
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke({"question": "What language is spoken in the Netherlands?"}))

### Try asking about something absent

- Ask a question the documents do not cover
- A model that answers anyway is the failure mode RAG is supposed to prevent,
  and the fix is in the prompt : tell it to say when the context does not cover
  the question